# Order Analytics — Interactive Spark + Glue Catalog
Reads from Glue external table, runs aggregations, writes Parquet to S3.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, round as _round, to_date

BUCKET        = "YOUR_BUCKET"
AWS_ACCOUNT   = "YOUR_AWS_ACCOUNT_ID"
GLUE_DB       = "ecommerce_db"
GLUE_TABLE    = "orders"
S3_OUTPUT     = f"s3a://{BUCKET}/curated/order_analytics/"

JAR_DIR = "/home/jovyan/jars"
jars = ",".join([
    f"{JAR_DIR}/hadoop-aws-3.3.4.jar",
    f"{JAR_DIR}/aws-java-sdk-bundle-1.12.262.jar",
    f"{JAR_DIR}/aws-glue-datacatalog-spark-client-3.4.0.jar",
])

spark = (
    SparkSession.builder
    .appName("OrderAnalyticsNotebook")
    .master("spark://spark-master:7077")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "2g")
    .config("spark.jars", jars)
    .config("spark.sql.catalogImplementation", "hive")
    .config("hive.metastore.client.factory.class",
            "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
    .config("spark.hadoop.hive.metastore.glue.catalogid", AWS_ACCOUNT)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("Spark UI:      http://localhost:4040")

In [ ]:
# Preview the Glue external table
orders = spark.sql(f"SELECT * FROM {GLUE_DB}.{GLUE_TABLE}")
orders = orders.withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
print(f"Row count: {orders.count()}")
orders.show(5)

In [ ]:
# Revenue by category
revenue_by_category = (
    orders.groupBy("category")
    .agg(
        _round(_sum(col("quantity") * col("unit_price")), 2).alias("total_revenue"),
        count("order_id").alias("total_orders")
    )
    .orderBy(col("total_revenue").desc())
)
revenue_by_category.show()

In [ ]:
# Top 10 products by revenue
top_products = (
    orders.groupBy("product_id", "product_name")
    .agg(_round(_sum(col("quantity") * col("unit_price")), 2).alias("revenue"))
    .orderBy(col("revenue").desc())
    .limit(10)
)
top_products.show()

In [ ]:
# Daily order trends
daily_trends = (
    orders.groupBy("order_date")
    .agg(
        count("order_id").alias("num_orders"),
        _round(_sum(col("quantity") * col("unit_price")), 2).alias("daily_revenue")
    )
    .orderBy("order_date")
)
daily_trends.show(10)

In [ ]:
# Write all results to S3 as Parquet
revenue_by_category.write.mode("overwrite").parquet(S3_OUTPUT + "revenue_by_category/")
top_products.write.mode("overwrite").parquet(S3_OUTPUT + "top_products/")
daily_trends.write.mode("overwrite").parquet(S3_OUTPUT + "daily_trends/")
print("Done. Written to:", S3_OUTPUT)

In [ ]:
spark.stop()